# Attention Mechanisms and Transformers with Grilly

**From self-attention to full transformer blocks with FlashAttention2 and RoPE.**

This notebook covers:
1. GPU/CPU detection
2. Self-attention explained mathematically
3. Using `nn.MultiheadAttention` on a sequence
4. Standard attention vs FlashAttention2
5. Rotary Positional Embeddings (RoPE)
6. Building a mini transformer block: attention + FFN + LayerNorm
7. PerceiverIO cross-attention example

We use small dimensions (d=64, heads=4) throughout to keep things fast
and avoid OOM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

import grilly
from grilly import nn
from grilly.nn import Variable, tensor
from grilly.optim import AdamW
import grilly.functional as F

# --- Backend detection ---
try:
    from grilly._bridge import is_vulkan_available
    DEVICE = "vulkan" if is_vulkan_available() else "cpu"
except (ImportError, AttributeError):
    DEVICE = "cpu"

print(f"grilly {grilly.__version__} | backend: {DEVICE}")
np.random.seed(42)

## 1. Self-Attention Explained

Self-attention allows each position in a sequence to attend to every other
position. Given an input sequence $X \in \mathbb{R}^{n \times d}$:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Where:
- $Q$ (queries), $K$ (keys), $V$ (values) are linear projections
- $d_k$ is the key dimension (used for scaling)
- The softmax produces attention weights: how much each position attends to others

Let's implement this from scratch to build intuition.

In [ ]:
def manual_attention(Q, K, V):
    """Compute scaled dot-product attention manually.

    Args:
        Q: Queries, shape (seq_len, d_k)
        K: Keys, shape (seq_len, d_k)
        V: Values, shape (seq_len, d_v)

    Returns:
        output: shape (seq_len, d_v)
        weights: shape (seq_len, seq_len) -- attention weights
    """
    d_k = Q.shape[-1]
    # Compute attention scores
    scores = Q @ K.T / np.sqrt(d_k)  # (seq_len, seq_len)
    # Softmax over keys dimension
    scores_exp = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = scores_exp / scores_exp.sum(axis=-1, keepdims=True)
    # Weighted sum of values
    output = weights @ V  # (seq_len, d_v)
    return output, weights

# Create a small example: 6-token sequence, d=8
seq_len = 6
d_model = 8

# Random input sequence
X = np.random.randn(seq_len, d_model).astype(np.float32)

# Random projection matrices (in practice, these are learned)
W_Q = np.random.randn(d_model, d_model).astype(np.float32) * 0.1
W_K = np.random.randn(d_model, d_model).astype(np.float32) * 0.1
W_V = np.random.randn(d_model, d_model).astype(np.float32) * 0.1

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

output, attn_weights = manual_attention(Q, K, V)

print(f"Input shape  : {X.shape}  (seq_len={seq_len}, d={d_model})")
print(f"Output shape : {output.shape}")
print(f"Weights shape: {attn_weights.shape}  (seq_len x seq_len)")
print()
print("Attention weights (each row sums to 1):")
print(np.round(attn_weights, 3))
print(f"\nRow sums: {np.round(attn_weights.sum(axis=-1), 4)}")

In [ ]:
# Visualize the attention pattern
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
im = ax.imshow(attn_weights, cmap='Blues', vmin=0, vmax=attn_weights.max())
ax.set_xlabel('Key Position')
ax.set_ylabel('Query Position')
ax.set_title('Self-Attention Weights')
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{attn_weights[i, j]:.2f}', ha='center', va='center',
                fontsize=8, color='black' if attn_weights[i, j] < 0.3 else 'white')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 2. Multi-Head Attention with `nn.MultiheadAttention`

Multi-head attention runs `h` parallel attention heads, each with its own
Q/K/V projections, then concatenates and projects the results.

Grilly's `nn.MultiheadAttention` handles all of this:
- Input shape: `(batch, seq_len, d_model)`
- Output shape: `(batch, seq_len, d_model)`

In [ ]:
# Configuration (small to avoid OOM)
batch_size = 4
seq_len = 16
d_model = 64
n_heads = 4
d_head = d_model // n_heads  # 16

print(f"Config: batch={batch_size}, seq_len={seq_len}, d_model={d_model}, heads={n_heads}, d_head={d_head}")
print()

# Create multi-head attention module
mha = nn.MultiheadAttention(d_model=d_model, n_heads=n_heads)

# Random input sequence
x_np = np.random.randn(batch_size, seq_len, d_model).astype(np.float32) * 0.1
x = Variable(x_np)

# Self-attention: query = key = value = x
output = mha(x, x, x)

print(f"Input shape : {x.data.shape}")
print(f"Output shape: {output.data.shape}")
print(f"\nOutput matches input shape (residual-compatible): {x.data.shape == output.data.shape}")

## 3. Standard Attention vs FlashAttention2

FlashAttention2 is an IO-aware attention algorithm that avoids materializing
the full $N \times N$ attention matrix, reducing memory from $O(N^2)$ to
$O(N)$. Grilly provides `F.flash_attention2` for this.

Let's compare:
- **Standard**: `softmax(QK^T / sqrt(d)) V` -- materializes full attention matrix
- **Flash**: tiled computation, same result, less memory

In [ ]:
# Generate Q, K, V for comparison
seq_len_test = 32
d_test = 64

Q_np = np.random.randn(batch_size, n_heads, seq_len_test, d_head).astype(np.float32) * 0.1
K_np = np.random.randn(batch_size, n_heads, seq_len_test, d_head).astype(np.float32) * 0.1
V_np = np.random.randn(batch_size, n_heads, seq_len_test, d_head).astype(np.float32) * 0.1

Q_var = Variable(Q_np)
K_var = Variable(K_np)
V_var = Variable(V_np)

# --- Standard attention ---
t0 = time.perf_counter()
# Compute scores: (batch, heads, seq, seq)
scores = np.einsum('bhqd,bhkd->bhqk', Q_np, K_np) / np.sqrt(d_head)
# Softmax
scores_exp = np.exp(scores - scores.max(axis=-1, keepdims=True))
attn_weights_std = scores_exp / scores_exp.sum(axis=-1, keepdims=True)
# Weighted values
output_standard = np.einsum('bhqk,bhkd->bhqd', attn_weights_std, V_np)
t_standard = time.perf_counter() - t0

print(f"Standard attention:")
print(f"  Output shape    : {output_standard.shape}")
print(f"  Attention matrix: {attn_weights_std.shape} ({attn_weights_std.size * 4 / 1024:.1f} KB)")
print(f"  Time            : {t_standard*1000:.2f} ms")
print()

# --- FlashAttention2 ---
t0 = time.perf_counter()
output_flash = F.flash_attention2(Q_var, K_var, V_var)
t_flash = time.perf_counter() - t0

print(f"FlashAttention2:")
print(f"  Output shape    : {output_flash.data.shape}")
print(f"  No attn matrix  : O(N) memory instead of O(N^2)")
print(f"  Time            : {t_flash*1000:.2f} ms")
print()

# Compare outputs (should be numerically close)
diff = np.abs(output_standard - output_flash.data)
print(f"Max absolute difference: {diff.max():.6f}")
print(f"Mean absolute difference: {diff.mean():.6f}")
print(f"Outputs match: {diff.max() < 1e-3}")

## 4. Rotary Positional Embeddings (RoPE)

RoPE encodes position by rotating Q and K vectors. Unlike absolute positional
embeddings, RoPE:
- Makes attention naturally relative (depends on position *difference*)
- Decays gracefully with distance
- Requires no additional learned parameters

The rotation applies a 2D rotation matrix to consecutive pairs of dimensions.

In [ ]:
def compute_rope_freqs(d_head, seq_len, base=10000.0):
    """Compute RoPE frequency matrix.

    Args:
        d_head: Dimension per head (must be even)
        seq_len: Maximum sequence length
        base: Base frequency

    Returns:
        cos, sin: Both shape (seq_len, d_head)
    """
    # Frequency for each dimension pair
    freqs = 1.0 / (base ** (np.arange(0, d_head, 2, dtype=np.float32) / d_head))
    # Position indices
    positions = np.arange(seq_len, dtype=np.float32)
    # Outer product: (seq_len, d_head/2)
    angles = np.outer(positions, freqs)
    # Repeat for pairs: (seq_len, d_head)
    cos_vals = np.cos(np.repeat(angles, 2, axis=1))
    sin_vals = np.sin(np.repeat(angles, 2, axis=1))
    return cos_vals.astype(np.float32), sin_vals.astype(np.float32)


def apply_rope(x, cos, sin):
    """Apply rotary embeddings to queries or keys.

    Args:
        x: Input tensor, shape (..., seq_len, d_head)
        cos, sin: Frequency matrices, shape (seq_len, d_head)

    Returns:
        Rotated tensor, same shape as x
    """
    # Rotate pairs: [x0, x1, x2, x3, ...] -> [-x1, x0, -x3, x2, ...]
    x_rotated = np.stack([-x[..., 1::2], x[..., ::2]], axis=-1)
    x_rotated = x_rotated.reshape(x.shape)
    return x * cos + x_rotated * sin


# Demonstrate RoPE
d_rope = 16
sl_rope = 32

cos_rope, sin_rope = compute_rope_freqs(d_rope, sl_rope)

print(f"RoPE config: d_head={d_rope}, seq_len={sl_rope}")
print(f"cos shape: {cos_rope.shape}")
print(f"sin shape: {sin_rope.shape}")

# Show how position 0 and position 10 rotate differently
q_sample = np.random.randn(1, sl_rope, d_rope).astype(np.float32)
q_rotated = apply_rope(q_sample, cos_rope, sin_rope)

print(f"\nOriginal Q[0, :4]: {np.round(q_sample[0, 0, :4], 3)}")
print(f"Rotated  Q[0, :4]: {np.round(q_rotated[0, 0, :4], 3)}")
print(f"Original Q[10,:4]: {np.round(q_sample[0, 10, :4], 3)}")
print(f"Rotated  Q[10,:4]: {np.round(q_rotated[0, 10, :4], 3)}")

In [ ]:
# Visualize RoPE frequencies
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cosine frequencies across positions
im0 = axes[0].imshow(cos_rope.T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Dimension')
axes[0].set_title('RoPE Cosine Frequencies')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Sine frequencies
im1 = axes[1].imshow(sin_rope.T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Dimension')
axes[1].set_title('RoPE Sine Frequencies')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.suptitle('Rotary Positional Embeddings', fontsize=13)
plt.tight_layout()
plt.show()

print("Low dimensions oscillate slowly (capture long-range position).")
print("High dimensions oscillate fast (capture fine-grained position).")

## 5. Mini Transformer Block

A transformer block combines:
1. **Multi-head self-attention** (with residual connection)
2. **LayerNorm**
3. **Feed-forward network (FFN)** with GELU activation
4. **Another LayerNorm** (Pre-LN architecture)

```
x -> LayerNorm -> MHA -> + x -> LayerNorm -> FFN -> + x -> output
```

In [ ]:
class TransformerBlock:
    """A single Pre-LN transformer block."""

    def __init__(self, d_model, n_heads, d_ff=None):
        if d_ff is None:
            d_ff = 4 * d_model  # Standard expansion ratio

        # Multi-head self-attention
        self.mha = nn.MultiheadAttention(d_model=d_model, n_heads=n_heads)
        self.ln1 = nn.LayerNorm(d_model)

        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)

    def __call__(self, x):
        # Pre-LN: normalize before attention
        normed = self.ln1(x)
        attn_out = self.mha(normed, normed, normed)
        x = x + attn_out  # Residual connection

        # Pre-LN: normalize before FFN
        normed = self.ln2(x)
        ffn_out = self.ffn(normed)
        x = x + ffn_out  # Residual connection

        return x

    def parameters(self):
        params = list(self.mha.parameters())
        params += list(self.ln1.parameters())
        params += list(self.ffn.parameters())
        params += list(self.ln2.parameters())
        return params


# Create a mini transformer block
d_model = 64
n_heads = 4
d_ff = 128  # Smaller than 4x for efficiency in this demo

block = TransformerBlock(d_model=d_model, n_heads=n_heads, d_ff=d_ff)

total_params = sum(p.data.size for p in block.parameters())
print(f"Transformer Block:")
print(f"  d_model  : {d_model}")
print(f"  n_heads  : {n_heads}")
print(f"  d_ff     : {d_ff}")
print(f"  Parameters: {total_params:,}")
print()

# Forward pass
batch_size = 4
seq_len = 16
x_np = np.random.randn(batch_size, seq_len, d_model).astype(np.float32) * 0.1
x = Variable(x_np)

output = block(x)
print(f"Input shape : {x.data.shape}")
print(f"Output shape: {output.data.shape}")
print(f"Shapes match: {x.data.shape == output.data.shape}")

## 6. Stacking Multiple Blocks

Real transformers stack multiple blocks. Let's create a small 3-block
transformer and run a forward + backward pass.

In [ ]:
class MiniTransformer:
    """A small transformer with multiple stacked blocks."""

    def __init__(self, n_layers, d_model, n_heads, d_ff=None):
        self.blocks = [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        self.final_ln = nn.LayerNorm(d_model)

    def __call__(self, x):
        for block in self.blocks:
            x = block(x)
        return self.final_ln(x)

    def parameters(self):
        params = []
        for block in self.blocks:
            params += block.parameters()
        params += list(self.final_ln.parameters())
        return params


# Create a 3-layer mini transformer
transformer = MiniTransformer(n_layers=3, d_model=64, n_heads=4, d_ff=128)

total_params = sum(p.data.size for p in transformer.parameters())
print(f"Mini Transformer: 3 layers, d=64, 4 heads")
print(f"Total parameters: {total_params:,}")
print()

# Forward pass
x = Variable(np.random.randn(2, 16, 64).astype(np.float32) * 0.1)
output = transformer(x)
print(f"Input  : {x.data.shape}")
print(f"Output : {output.data.shape}")
print()

# Backward pass
loss = (output * output).sum() / output.data.size
loss.backward()

# Verify gradients exist
n_grads = sum(1 for p in transformer.parameters() if p.grad is not None)
n_total = len(transformer.parameters())
print(f"Parameters with gradients: {n_grads}/{n_total}")
print(f"Backward pass successful: {n_grads > 0}")

## 7. PerceiverIO Cross-Attention

PerceiverIO uses **cross-attention** to map from a large input sequence
to a smaller set of latent vectors, then processes them with self-attention,
and decodes back to the desired output.

Key insight: by using a small number of latents, you avoid the $O(N^2)$
cost of self-attention on the full input.

```
Input (long)  ->
                 Cross-Attention  -> Latents (short) -> Self-Attention -> 
Latents (init) ->                                                         
                                                                          
                 Cross-Attention  -> Output
Output query  ->
```

In [ ]:
class PerceiverBlock:
    """Simplified PerceiverIO: cross-attend input -> latents, self-attend latents."""

    def __init__(self, d_model, n_heads, n_latents):
        self.n_latents = n_latents

        # Learned latent array
        self.latents = Variable(
            np.random.randn(1, n_latents, d_model).astype(np.float32) * 0.02
        )

        # Cross-attention: latents attend to input
        self.cross_attn = nn.MultiheadAttention(d_model=d_model, n_heads=n_heads)
        self.ln_cross = nn.LayerNorm(d_model)

        # Self-attention on latents
        self.self_attn = nn.MultiheadAttention(d_model=d_model, n_heads=n_heads)
        self.ln_self = nn.LayerNorm(d_model)

        # FFN
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Linear(d_model * 2, d_model),
        )
        self.ln_ffn = nn.LayerNorm(d_model)

    def __call__(self, x_input):
        batch_size = x_input.data.shape[0]

        # Expand latents to batch size
        latents = Variable(np.broadcast_to(
            self.latents.data, (batch_size, self.n_latents, x_input.data.shape[-1])
        ).copy())

        # Cross-attention: Q=latents, K=V=input
        normed_latents = self.ln_cross(latents)
        cross_out = self.cross_attn(normed_latents, x_input, x_input)
        latents = latents + cross_out

        # Self-attention on latents
        normed = self.ln_self(latents)
        self_out = self.self_attn(normed, normed, normed)
        latents = latents + self_out

        # FFN
        normed = self.ln_ffn(latents)
        ffn_out = self.ffn(normed)
        latents = latents + ffn_out

        return latents

    def parameters(self):
        params = [self.latents]
        params += list(self.cross_attn.parameters())
        params += list(self.ln_cross.parameters())
        params += list(self.self_attn.parameters())
        params += list(self.ln_self.parameters())
        params += list(self.ffn.parameters())
        params += list(self.ln_ffn.parameters())
        return params


# Create PerceiverIO block
d_model = 64
n_heads = 4
n_latents = 8   # Much smaller than input sequence

perceiver = PerceiverBlock(d_model=d_model, n_heads=n_heads, n_latents=n_latents)

total_params = sum(p.data.size for p in perceiver.parameters())
print(f"PerceiverIO Block:")
print(f"  d_model     : {d_model}")
print(f"  n_heads     : {n_heads}")
print(f"  n_latents   : {n_latents}")
print(f"  Parameters  : {total_params:,}")
print()

# Process a long input sequence through the bottleneck
input_seq_len = 128  # Long input
x_long = Variable(np.random.randn(2, input_seq_len, d_model).astype(np.float32) * 0.1)

latent_output = perceiver(x_long)

print(f"Input  : {x_long.data.shape}  (batch=2, seq={input_seq_len}, d={d_model})")
print(f"Output : {latent_output.data.shape}  (batch=2, latents={n_latents}, d={d_model})")
print()
print(f"Compression ratio: {input_seq_len}:{n_latents} = {input_seq_len/n_latents:.0f}x")
print(f"Self-attention cost: O({n_latents}^2) vs O({input_seq_len}^2)")
print(f"Speedup factor: {(input_seq_len**2) / (n_latents**2):.0f}x")

## 8. Putting It All Together: Attention Comparison

Let's visualize how different attention patterns look for a sequence.

In [ ]:
# Generate some example attention patterns for visualization
sl = 12  # Sequence length for visualization

# 1. Full self-attention (dense)
full_attn = np.random.rand(sl, sl).astype(np.float32)
full_attn = np.exp(full_attn) / np.exp(full_attn).sum(axis=-1, keepdims=True)

# 2. Causal (autoregressive) attention
causal_mask = np.tril(np.ones((sl, sl)))
causal_scores = np.random.rand(sl, sl).astype(np.float32) * causal_mask
causal_scores[causal_mask == 0] = -1e9
causal_exp = np.exp(causal_scores - causal_scores.max(axis=-1, keepdims=True))
causal_attn = causal_exp / causal_exp.sum(axis=-1, keepdims=True)
causal_attn[causal_mask == 0] = 0

# 3. Cross-attention (latents attend to input)
n_lat = 4
cross_scores = np.random.rand(n_lat, sl).astype(np.float32)
cross_attn = np.exp(cross_scores) / np.exp(cross_scores).sum(axis=-1, keepdims=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im0 = axes[0].imshow(full_attn, cmap='Blues', vmin=0)
axes[0].set_title('Full Self-Attention')
axes[0].set_xlabel('Key')
axes[0].set_ylabel('Query')

im1 = axes[1].imshow(causal_attn, cmap='Blues', vmin=0)
axes[1].set_title('Causal (Autoregressive)')
axes[1].set_xlabel('Key')
axes[1].set_ylabel('Query')

im2 = axes[2].imshow(cross_attn, cmap='Blues', vmin=0, aspect='auto')
axes[2].set_title(f'Cross-Attention ({n_lat} latents)')
axes[2].set_xlabel(f'Input ({sl} tokens)')
axes[2].set_ylabel(f'Latents ({n_lat})')

for ax in axes:
    ax.grid(False)

plt.suptitle('Attention Pattern Comparison', fontsize=13)
plt.tight_layout()
plt.show()

print("Full attention: every position attends to every other (bidirectional).")
print("Causal attention: each position only attends to past positions.")
print("Cross-attention: a small set of latents attends to the full input.")

## Summary

In this notebook you learned:

- **Self-attention**: $\text{softmax}(QK^T/\sqrt{d_k})V$ -- each position
  attends to all others
- **Multi-head attention**: parallel heads capture different relationships
- **FlashAttention2**: same output, $O(N)$ memory instead of $O(N^2)$
- **RoPE**: rotary positional embeddings encode position via rotation --
  no learned parameters, naturally relative
- **Transformer block**: attention + FFN + LayerNorm with residual connections
- **PerceiverIO**: cross-attention bottleneck compresses long sequences
  into a small latent set

### Key Takeaways
- Grilly's `nn.MultiheadAttention` and `F.flash_attention2` handle the
  heavy lifting with automatic GPU/CPU fallback
- Small dimensions (d=64, heads=4) keep demos fast -- scale up for
  production workloads
- Pre-LN transformers are more stable to train than Post-LN
- Cross-attention (PerceiverIO) is a powerful pattern for reducing
  computational cost on long sequences